# OpenInsight Data Ingestion Pipeline - Kaggle Edition

This notebook runs the full OpenInsight ingestion pipeline on Kaggle GPU and writes data to Zilliz Cloud, MongoDB Atlas, and Kaggle working storage.

## Where to put API keys

In Kaggle, open the notebook sidebar and add each secret under **Add-ons -> Secrets**. Use the exact secret names below so the notebook can read them automatically:

- `ZILLIZ_URI`
- `ZILLIZ_TOKEN`
- `MONGODB_URL`
- `MONGODB_DB`
- `NVIDIA_NIM_API_KEY`
- `HF_API_TOKEN`  
- `COHERE_API_KEY`  
- `NCBI_API_KEY`  
- `NCBI_EMAIL`

Optional settings you can also add as secrets if you want to override the defaults:

- `VECTOR_COLLECTION_V2`
- `VECTOR_COLLECTION`
- `VECTOR_DIM`
- `EMBED_PROVIDER`
- `RERANK_PROVIDER`
- `DENSE_MODEL_NAME`
- `RERANKER_MODEL_NAME`
- `PARSING_THREAD_WORKERS`
- `INGESTION_THREAD_WORKERS`

## How to run it

1. Upload your PDFs or XML files as a Kaggle dataset.
2. Set `DATA_DIR` in the configuration cell to the mounted dataset path, for example `/kaggle/input/your-dataset-name`.
3. Run the notebook cells from top to bottom.
4. Start by installing dependencies, then configure secrets, then launch GROBID.
5. Run the ingestion cell to process files and write to Zilliz Cloud and MongoDB.
6. Run the final verification cell to confirm documents, chunks, checkpoints, and metrics were saved.

## Notes

- `EMBED_PROVIDER=local` and `RERANK_PROVIDER=local` are for Kaggle GPU ingestion.
- If you are using the same embeddings at query time on CPU, switch the query-side environment to the remote provider you configured in the repo.
- Kaggle input folders are read-only, so do not try to write inside `/kaggle/input`.
- Persistent files should go under `/kaggle/working`.


## 1. Install Dependencies

Install the same runtime packages used by the repo, plus Java and Tesseract for GROBID/OCR support.

In [ ]:
# ── Fix NumPy ABI compatibility (Kaggle ships numpy 2.x, pymilvus needs 1.x) ──
# Must force-reinstall pandas AFTER downgrading numpy so its C extensions
# are rebuilt against the correct numpy 1.x headers.
%pip install -q 'numpy<2.0'
%pip install -q --force-reinstall --no-deps pandas

# Install OpenInsight dependencies
%pip install -q \
    fastapi uvicorn python-dotenv pydantic pydantic-settings \
    pymongo motor pypdf2 pdfplumber beautifulsoup4 requests httpx lxml biopython \
    sentence-transformers transformers torch \
    'pymilvus>=2.5,<2.6' langchain langchain-community openai redis \
    tqdm loguru tenacity python-slugify pytesseract Pillow scispacy

# Verify numpy/pandas/pymilvus compatibility
import numpy as np
print(f'numpy: {np.__version__}')
import pandas as pd
print(f'pandas: {pd.__version__}')
from pymilvus import MilvusClient
print('pymilvus: OK')

# Kaggle system packages for GROBID and OCR
!apt-get update -qq
!apt-get install -qq -y default-jre wget unzip tesseract-ocr > /dev/null 2>&1

import os
import sys
from pathlib import Path

REPO_DIR = Path('/kaggle/working/openinsight')
if REPO_DIR.exists():
    %cd /kaggle/working/openinsight
    !git pull origin restruct
else:
    !git clone -b restruct https://github.com/Adi103-ETAI/openinsight.git /kaggle/working/openinsight

sys.path.insert(0, str(REPO_DIR))
print('Environment ready.')


## 2. Configure Kaggle Secrets and Environment

Use Kaggle Secrets for API keys. The notebook mirrors the repository settings surface so the pipeline runs with the same configuration logic.

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
from pathlib import Path


def get_secret(name: str, default: str = '') -> str:
    try:
        client = UserSecretsClient()
        value = client.get_secret(name)
        return default if value is None else value
    except Exception:
        return default


# API holders / service secrets
ZILLIZ_URI         = get_secret('ZILLIZ_URI', '')
ZILLIZ_TOKEN       = get_secret('ZILLIZ_TOKEN', '')
MONGODB_URL        = get_secret('MONGODB_URL', '')
MONGODB_DB         = get_secret('MONGODB_DB', 'openinsight')
NVIDIA_NIM_API_KEY = get_secret('NVIDIA_NIM_API_KEY', '')
HF_API_TOKEN       = get_secret('HF_API_TOKEN', '')
COHERE_API_KEY     = get_secret('COHERE_API_KEY', '')
NCBI_API_KEY       = get_secret('NCBI_API_KEY', '')
NCBI_EMAIL         = get_secret('NCBI_EMAIL', 'sentarc.ai@gmail.com')

# Ingestion target
AVAILABLE_SOURCES = ['pubmed', 'icmr', 'cochrane', 'nmc_guideline', 'rssdi', 'who', 'cdc', 'statpearls']
SOURCE_TYPE = 'pubmed'  # change this as needed
DATA_DIR = '/kaggle/input/your-dataset-name'  # UPDATE THIS to your Kaggle dataset path
BATCH_SIZE = 10
RECREATE_INDEX = False
RESUME = True
RESET = False
SINGLE_FILE = None

# ── Environment variables for the pipeline ──
# Kaggle has GPU, so use local providers for ingestion
os.environ['MONGODB_URL']            = MONGODB_URL
os.environ['MONGODB_DB']             = MONGODB_DB
os.environ['VECTOR_URI']             = ZILLIZ_URI
os.environ['VECTOR_TOKEN']           = ZILLIZ_TOKEN
os.environ['MILVUS_CLOUD']           = 'true'
os.environ['VECTOR_COLLECTION_V2']   = 'openinsight_v2'
os.environ['NVIDIA_NIM_API_KEY']     = NVIDIA_NIM_API_KEY
os.environ['NVIDIA_NIM_BASE_URL']    = 'https://integrate.api.nvidia.com/v1'
os.environ['NIM_MODEL']              = 'meta/llama-3.1-70b-instruct'
os.environ['EMBED_PROVIDER']         = 'local'   # Kaggle has GPU -> use SentenceTransformers locally
os.environ['RERANK_PROVIDER']        = 'local'   # Kaggle has GPU -> use CrossEncoder locally
os.environ['HF_API_TOKEN']           = HF_API_TOKEN  # Needed for gated model download
os.environ['DENSE_MODEL_NAME']       = 'pritamdeka/S-PubMedBert-MS-MARCO'
os.environ['HF_EMBED_MODEL']         = 'pritamdeka/S-PubMedBert-MS-MARCO'
os.environ['RERANKER_MODEL_NAME']    = 'BAAI/bge-reranker-v2-m3'
os.environ['COHERE_API_KEY']         = COHERE_API_KEY
os.environ['NCBI_API_KEY']           = NCBI_API_KEY
os.environ['NCBI_EMAIL']             = NCBI_EMAIL
os.environ['SSL_CERT_FILE']          = __import__('certifi').where()

# Optional: StatPearls query list for PubMed seeding
STATPEARLS_QUERIES = [
    'StatPearls[book] diabetes mellitus',
    'StatPearls[book] hypertension management',
    'StatPearls[book] acute coronary syndrome',
    'StatPearls[book] sepsis management',
    'StatPearls[book] pneumonia treatment',
    'StatPearls[book] tuberculosis',
    'StatPearls[book] dengue fever',
    'StatPearls[book] malaria treatment',
    'StatPearls[book] renal failure acute',
    'StatPearls[book] stroke management',
    'StatPearls[book] anemia iron deficiency',
    'StatPearls[book] liver cirrhosis',
    'StatPearls[book] epilepsy seizure',
    'StatPearls[book] asthma COPD',
    'StatPearls[book] obstetric emergency',
]

Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

print('Configuration loaded.')
print(f'Source type: {SOURCE_TYPE}')
print(f'Available sources: {", ".join(AVAILABLE_SOURCES)}')
print(f'Data directory: {DATA_DIR}')
print(f'Zilliz URI set: {bool(ZILLIZ_URI)}')
print(f'MongoDB URL set: {bool(MONGODB_URL)}')
print(f'NVIDIA NIM API key set: {bool(NVIDIA_NIM_API_KEY)}')
print(f'Embed provider: {os.environ["EMBED_PROVIDER"]}')
print(f'Rerank provider: {os.environ["RERANK_PROVIDER"]}')
print(f'StatPearls queries: {len(STATPEARLS_QUERIES)}')


## 3. Start GROBID and Verify GPU

Kaggle does not ship with GROBID. This cell starts it locally and confirms GPU availability.

In [ ]:
# Start GROBID for PDF parsing and verify GPU availability
import subprocess
import time
import httpx
import torch

GROBID_DIR = Path('/kaggle/working/grobid')
GROBID_BIN = GROBID_DIR / 'grobid-0.9.0' / 'bin' / 'grobid-server'
GROBID_URL = 'http://localhost:8070'

if not GROBID_BIN.exists():
    !wget -q https://github.com/grobidOrg/grobid/releases/download/0.9.0/grobid-0.9.0.zip -O /kaggle/working/grobid-0.9.0.zip
    !unzip -q -o /kaggle/working/grobid-0.9.0.zip -d /kaggle/working/grobid

if 'grobid_proc' not in globals() or grobid_proc.poll() is not None:
    grobid_proc = subprocess.Popen(
        [str(GROBID_BIN), '--port', '8070'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

os.environ['GROBID_URL'] = GROBID_URL

# Enhanced health check with fallback for different GROBID versions
for attempt in range(5):
    try:
        # Try new health endpoint first (GROBID 0.9.0+)
        response = httpx.get(f'{GROBID_URL}/api/health', timeout=10)
        if response.status_code == 200:
            print('GROBID 0.9.0: ready (health endpoint)')
            break
    except:
        pass
    # Fallback to isalive for older GROBID versions
    try:
        response = httpx.get(f'{GROBID_URL}/api/isalive', timeout=10)
        if response.status_code == 200:
            print('GROBID: ready (isalive fallback)')
            break
    except:
        pass
    time.sleep(3)
else:
    print('Warning: GROBID not responding. Will use fallback parsers.')

print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print('Configured input directories:')
print(f'  Repo: {REPO_DIR}')
print(f'  Data: {DATA_DIR}')

## 3.5. Wipe Old Data (First Run Only)

* Run this cell **once** before your first ingestion to clear any old/bad data.
* Skip this on subsequent runs to keep existing data and use resume.
* This deletes the Zilliz collection and all MongoDB collections.

In [ ]:
# ── WIPE ALL DATA (Run this once before first ingestion) ──
# This clears old bad data from Zilliz Cloud and MongoDB.
# After your first successful run, you can skip this cell.
from pymilvus import MilvusClient
from pymongo import MongoClient

# Wipe Zilliz Cloud collection
zilliz = MilvusClient(uri=ZILLIZ_URI, token=ZILLIZ_TOKEN)
if zilliz.has_collection('openinsight_v2'):
    zilliz.drop_collection('openinsight_v2')
    print('Zilliz: Dropped collection openinsight_v2')
else:
    print('Zilliz: Collection does not exist yet')

# Wipe all MongoDB collections
mongo = MongoClient(MONGODB_URL)
db = mongo[MONGODB_DB]
for col_name in ['documents_v2', 'chunks_v2', 'failed_documents',
                  'ingestion_checkpoints', 'ingestion_metrics']:
    result = db[col_name].delete_many({})
    print(f'MongoDB {col_name}: Deleted {result.deleted_count} docs')

print('\nAll data wiped! Ready for fresh ingestion.')

## 4. Run Ingestion

This cell uses the repository's actual ingestion pipeline, 
* including deduplication,
* enrichment,
* chunking,
* quality scoring,
* embeddings,
* vector indexing, Mongo storage, and checkpointing.

In [ ]:
# Run the ingestion pipeline using the repo's native orchestrator
import asyncio
from pathlib import Path

from src.ingestion.pipeline import IngestionPipeline
from src.ingestion.run_ingestion import SOURCES


def list_candidate_files(directory: str) -> list[Path]:
    root = Path(directory)
    if not root.exists():
        return []
    return sorted(
        [
            path
            for path in root.rglob('*')
            if path.is_file() and path.suffix.lower() in {'.pdf', '.xml'}
        ]
    )


def print_source_catalog() -> None:
    print('Available sources:')
    for source in SOURCES:
        print(f'  - {source}')


print_source_catalog()
print(f'Candidate files in DATA_DIR: {len(list_candidate_files(DATA_DIR))}')

async def run_ingestion() -> dict:
    pipeline = IngestionPipeline()

    print('\nStarting ingestion with these settings:')
    print(f'  source: {SOURCE_TYPE}')
    print(f'  directory: {DATA_DIR}')
    print(f'  batch_size: {BATCH_SIZE}')
    print(f'  recreate_index: {RECREATE_INDEX}')
    print(f'  resume: {RESUME}')
    print(f'  reset: {RESET}')
    print(f'  single_file: {SINGLE_FILE}')
    print('=' * 60)

    summary = await pipeline.ingest_directory(
        directory=DATA_DIR,
        source=SOURCE_TYPE,
        recreate_index=RECREATE_INDEX,
        batch_size=BATCH_SIZE,
        resume=RESUME,
        reset=RESET,
        single_file=SINGLE_FILE,
    )

    print('\n' + '=' * 60)
    print('INGESTION COMPLETE')
    print('=' * 60)
    for key, value in summary.items():
        print(f'  {key}: {value}')

    recent_runs = await pipeline.monitor.get_recent_runs(limit=5)
    print('\nRecent ingestion runs:')
    for run in recent_runs:
        print(f"  - {run.get('source_type', 'unknown')} | {run.get('status', 'unknown')} | {run.get('documents_stored', 0)} docs | {run.get('chunks_embedded', 0)} chunks")

    return summary

summary = await run_ingestion()

## 5. Verify Storage and Metrics

Confirm the ingestion wrote to Milvus, MongoDB, checkpoints, and the run metrics collection.

In [ ]:
# Verify data written to Zilliz Cloud, MongoDB, checkpoints, and run metrics
from pymilvus import MilvusClient
from pymongo import MongoClient

from src.ingestion.monitoring import IngestionMonitor
from src.data.mongo.connection import get_mongo_db

collection_name = os.environ.get('VECTOR_COLLECTION_V2', 'openinsight_v2')

zilliz_client = MilvusClient(uri=ZILLIZ_URI, token=ZILLIZ_TOKEN)
if zilliz_client.has_collection(collection_name):
    stats = zilliz_client.get_collection_stats(collection_name)
    print(f'Zilliz collection: {collection_name}')
    print(f"Row count: {stats.get('row_count', 'unknown')}")
else:
    print(f'Collection not found: {collection_name}')

mongo_client = MongoClient(MONGODB_URL)
db = mongo_client[MONGODB_DB]
print(f'\nMongoDB database: {MONGODB_DB}')
print(f"documents_v2: {db.documents_v2.count_documents({})}")
print(f"chunks_v2: {db.chunks_v2.count_documents({})}")
print(f"failed_documents: {db.failed_documents.count_documents({})}")
print(f"ingestion_checkpoints: {db.ingestion_checkpoints.count_documents({})}")
print(f"ingestion_metrics: {db.ingestion_metrics.count_documents({})}")

# Use the repo's monitor helper for a more structured view of storage stats
mongo_db = get_mongo_db(MONGODB_DB)
monitor = IngestionMonitor(mongo_db)

async def show_monitoring() -> None:
    storage_stats = await monitor.get_storage_stats()
    print('\nStorage stats by source:')
    print(storage_stats)

    recent_runs = await monitor.get_recent_runs(limit=3)
    print('\nLatest run metrics:')
    for run in recent_runs:
        print(
            f"  - {run.get('source_type', 'unknown')} | "
            f"{run.get('status', 'unknown')} | "
            f"docs={run.get('documents_stored', 0)} | "
            f"chunks={run.get('chunks_embedded', 0)} | "
            f"duration={run.get('duration_seconds', 0)}s"
        )

await show_monitoring()

# Optional cleanup
try:
    grobid_proc.terminate()
    print('\nGROBID stopped.')
except Exception:
    pass